In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Агрегация дополнительных наборов данных

In [3]:
# Функция для загрузки данных
def load_data(file):
    path = '../../data/raw/'
    df = pd.read_csv(path + file)
    return df

# Загрузим данные
bureau = load_data('bureau.csv')
bb = load_data('bureau_balance.csv')
prev_app = load_data('previous_application.csv')
ccb = load_data('credit_card_balance.csv')
ip = load_data('installments_payments.csv')
pc = load_data('POS_CASH_balance.csv')

# Посмотрим на размеры наборов
print(f'Bureau: {bureau.shape}')
print(f'Bureau balance: {bb.shape}')
print(f'Previous application: {prev_app.shape}')
print(f'Credit card balance: {ccb.shape}')
print(f'Installments payments: {ip.shape}')
print(f'Pos cash: {pc.shape}')

Bureau: (1716428, 17)
Bureau balance: (27299925, 3)
Previous application: (1670214, 37)
Credit card balance: (3840312, 23)
Installments payments: (13605401, 8)
Pos cash: (10001358, 8)


## Bureau + bureau balance

In [4]:
# Объединим bureau и bureau balance по ключу ID записи в бюро
full_bureau = bureau.merge(bb, on='SK_ID_BUREAU', how='left')
full_bureau.shape

(25121815, 19)

In [5]:
full_bureau.SK_ID_CURR.value_counts() 

SK_ID_CURR
318065    2791
260762    2657
186249    2217
218517    2102
364882    1941
          ... 
443093       1
385789       1
162644       1
179516       1
448157       1
Name: count, Length: 305811, dtype: int64

Необходимо агрегировать признаки, чтобы объединять с основным набором, т.к. на ID клиента могут приходится несколько записей. Будем использовать следующие агрегации:
- частота на категориальных признаках или количество
- min, max, mean, sum на числовых признаков
- кастомные функции

In [6]:
# Выведем информацию об объединённом наборе данных
full_bureau.info()
print(full_bureau.CREDIT_ACTIVE.unique())
print(full_bureau.CREDIT_CURRENCY.unique())

<class 'pandas.DataFrame'>
RangeIndex: 25121815 entries, 0 to 25121814
Data columns (total 19 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_CURR              int64  
 1   SK_ID_BUREAU            int64  
 2   CREDIT_ACTIVE           str    
 3   CREDIT_CURRENCY         str    
 4   DAYS_CREDIT             int64  
 5   CREDIT_DAY_OVERDUE      int64  
 6   DAYS_CREDIT_ENDDATE     float64
 7   DAYS_ENDDATE_FACT       float64
 8   AMT_CREDIT_MAX_OVERDUE  float64
 9   CNT_CREDIT_PROLONG      int64  
 10  AMT_CREDIT_SUM          float64
 11  AMT_CREDIT_SUM_DEBT     float64
 12  AMT_CREDIT_SUM_LIMIT    float64
 13  AMT_CREDIT_SUM_OVERDUE  float64
 14  CREDIT_TYPE             str    
 15  DAYS_CREDIT_UPDATE      int64  
 16  AMT_ANNUITY             float64
 17  MONTHS_BALANCE          float64
 18  STATUS                  str    
dtypes: float64(9), int64(6), str(4)
memory usage: 3.6 GB
<StringArray>
['Closed', 'Active', 'Sold', 'Bad debt']
Length

In [7]:
# Переведём статус в числовую плоскость
statuses = full_bureau.STATUS.unique().tolist()
mapper = {
    'X': np.nan,
    'C': 0,  # закрыт
    '0': 0,  # нет просрочки
    '1': 1,  # 1–30 дней
    '2': 2,  # 31–60 дней
    '3': 3,  # 61–90 дней
    '4': 4,  # 91-120 дней
    '5': 5,  # > 120 дней
}

full_bureau['status_num'] = full_bureau['STATUS'].map(mapper)

In [9]:
# Агрегируем данные бюро
bureau_agg = full_bureau.groupby('SK_ID_CURR').agg(

    # Кол-во записей
    records_count = ('SK_ID_BUREAU', 'count'),

    # Активный кредит
    closed_credits = ('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),
    active_credits = ('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    sold_credits = ('CREDIT_ACTIVE', lambda x: (x == 'Sold').sum()),
    bad_credits = ('CREDIT_ACTIVE', lambda x: (x == 'Bad debt').sum()),

    # Тип валюты
    curr1 = ('CREDIT_CURRENCY', lambda x: (x == 'currency 1').sum()),
    curr2 = ('CREDIT_CURRENCY', lambda x: (x == 'currency 2').sum()),
    curr3 = ('CREDIT_CURRENCY', lambda x: (x == 'currency 3').sum()),
    curr4 = ('CREDIT_CURRENCY', lambda x: (x == 'currency 4').sum()),

    # Информация о днях до заявки
    min_days_credit = ('DAYS_CREDIT', 'min'),
    max_days_credit = ('DAYS_CREDIT', 'max'),
    max_enddate = ('DAYS_CREDIT_ENDDATE', 'max'),
    max_enddate_fact = ('DAYS_ENDDATE_FACT', 'max'),
    min_update = ('DAYS_CREDIT_UPDATE', 'min'),
    max_update = ('DAYS_CREDIT_UPDATE', 'max'),

    # Числовые признаки
    max_overdue = ('CREDIT_DAY_OVERDUE', 'max'),
    max_sum_overdue = ('AMT_CREDIT_MAX_OVERDUE', 'sum'),
    mean_sum_overdue = ('AMT_CREDIT_MAX_OVERDUE', 'mean'),
    sum_amt = ('AMT_CREDIT_SUM', 'sum'),
    sum_debt = ('AMT_CREDIT_SUM_DEBT', 'sum'),
    sum_limit = ('AMT_CREDIT_SUM_LIMIT', 'sum'),
    sum_amt_overdue = ('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    mean_amt = ('AMT_CREDIT_SUM_OVERDUE', 'mean'),
    sum_amt_annuity = ('AMT_ANNUITY', 'sum'),
    mean_amt_annuity = ('AMT_ANNUITY', 'mean'),
    sum_months_balance = ('MONTHS_BALANCE', 'sum'),
    mean_month_balance = ('MONTHS_BALANCE', 'mean'),

    # Агрегация статуса
    max_dpd = ('status_num', 'max'),
    mean_dpd = ('status_num', 'mean'),
    months_overdue = ('status_num', lambda x: (x > 0).sum()),  # кол-во месяцев с просрочкой
    ever_written_off = ('status_num', lambda x: (x == 5).any().astype(int)),  # хотя бы один статус 'bad'
    total_months = ('status_num', 'count')
)

bureau_agg.shape

(305811, 32)

SK_ID_CURR - ключ, который используется для группировки. Далее последовательно к признакам применяются агрегационные функции:
- кол-во записей в бюро
- частота по активному/закрытому кредиту
- частота по типу валют
- минимальное/максимальное количество дней до текущей заявки в Home Credit
- максимальная конечная и максимальная фактическая дата окончания кредита
- минимальная/максимальная дата обновления информация бюро до текущей заявки
- числовые признаки: максимальная, минимальная, средняя сумма просрочки, суммы кредитов, платежей, средний и суммарный баланс
- кол-во месяцев с просрочкой
- хотя бы один статус 'bad' по кредиту

In [10]:
# Сохраним данные
bureau_agg.to_csv('../../data/clean/bureau_agg.csv')

## Previous info

In [12]:
prev_app.SK_ID_CURR.value_counts()

SK_ID_CURR
187868    77
265681    73
173680    72
242412    68
206783    67
          ..
239799     1
174832     1
253940     1
353284     1
191629     1
Name: count, Length: 338857, dtype: int64

In [13]:
# Объединим другие наборы по ключу номера предыдущей заявки после агрегации
print(ip.info())
print(ip.head())

<class 'pandas.DataFrame'>
RangeIndex: 13605401 entries, 0 to 13605400
Data columns (total 8 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_PREV              int64  
 1   SK_ID_CURR              int64  
 2   NUM_INSTALMENT_VERSION  float64
 3   NUM_INSTALMENT_NUMBER   int64  
 4   DAYS_INSTALMENT         float64
 5   DAYS_ENTRY_PAYMENT      float64
 6   AMT_INSTALMENT          float64
 7   AMT_PAYMENT             float64
dtypes: float64(5), int64(3)
memory usage: 830.4 MB
None
   SK_ID_PREV  SK_ID_CURR  NUM_INSTALMENT_VERSION  NUM_INSTALMENT_NUMBER  \
0     1054186      161674                     1.0                      6   
1     1330831      151639                     0.0                     34   
2     2085231      193053                     2.0                      1   
3     2452527      199697                     1.0                      3   
4     2714724      167756                     1.0                      2   

   DAYS_INSTA

In [14]:
print(pc.info())
print(pc.NAME_CONTRACT_STATUS.unique())

<class 'pandas.DataFrame'>
RangeIndex: 10001358 entries, 0 to 10001357
Data columns (total 8 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   SK_ID_PREV             int64  
 1   SK_ID_CURR             int64  
 2   MONTHS_BALANCE         int64  
 3   CNT_INSTALMENT         float64
 4   CNT_INSTALMENT_FUTURE  float64
 5   NAME_CONTRACT_STATUS   str    
 6   SK_DPD                 int64  
 7   SK_DPD_DEF             int64  
dtypes: float64(2), int64(5), str(1)
memory usage: 610.4 MB
None
<StringArray>
[               'Active',             'Completed',                'Signed',
              'Approved', 'Returned to the store',                'Demand',
              'Canceled',                   'XNA',        'Amortized debt']
Length: 9, dtype: str


In [15]:
# Применяем ту же самую технику к installment payments, но по ключу предыдущих заявок

ip_agg = ip.groupby('SK_ID_PREV').agg(
    days_max = ('DAYS_INSTALMENT', 'max'),  # максимальная дата платежа
    days_entry_max = ('DAYS_ENTRY_PAYMENT', 'max'),  # фактическая максимальная дата платежа
    mean_installment = ('AMT_INSTALMENT', 'mean'),  # средняя установленная сумма платежа
    sum_installment = ('AMT_INSTALMENT', 'sum'),  # суммарная установленная  сумма платежа
    mean_payment = ('AMT_PAYMENT', 'mean'),  # средняя фактическая сумма платежа
    sum_payment = ('AMT_PAYMENT', 'sum')  # суммарная фактическая сумма платежа
)

print(f'Агрегация instalment payments: {ip_agg.shape}')

Агрегация instalment payments: (997752, 6)


In [16]:
# Сортируем, чтобы легче было брать последние значения
pc = pc.sort_values(['SK_ID_PREV', 'MONTHS_BALANCE'])

# Группировка и агрегация
pos_agg = pc.groupby('SK_ID_PREV').agg(

    # Количество месяцев в истории
    pos_months_count=('MONTHS_BALANCE', 'count'),

    # Минимальный (самый старый) и максимальный (самый свежий) месяц
    pos_first_month=('MONTHS_BALANCE', 'min'),
    pos_last_month=('MONTHS_BALANCE', 'max'),
    
    # Статистики по сроку кредита
    pos_instalment_last=('CNT_INSTALMENT', 'last'),
    pos_instalment_mean=('CNT_INSTALMENT', 'mean'),
    pos_instalment_max=('CNT_INSTALMENT', 'max'),
    
    # Статистики по оставшимся платежам
    pos_future_last=('CNT_INSTALMENT_FUTURE', 'last'),
    pos_future_mean=('CNT_INSTALMENT_FUTURE', 'mean'),
    pos_future_min=('CNT_INSTALMENT_FUTURE', 'min'),
    pos_future_max=('CNT_INSTALMENT_FUTURE', 'max'),

    # Сокращение оставшихся платежей от первого до последнего месяца
    pos_future_diff=('CNT_INSTALMENT_FUTURE', lambda x: x.iloc[-1] - x.iloc[0] if len(x) > 1 else 0),
    
    # Статистики по просрочкам SK_DPD
    pos_dpd_last=('SK_DPD', 'last'),
    pos_dpd_max=('SK_DPD', 'max'),
    pos_dpd_mean=('SK_DPD', 'mean'),
    pos_dpd_std=('SK_DPD', 'std'),
    pos_dpd_any_overdue=('SK_DPD', lambda x: (x > 0).any()),
    pos_dpd_pct_overdue=('SK_DPD', lambda x: (x > 0).mean()),
    
    # Аналогично для SK_DPD_DEF: просрочка с допуском
    pos_dpd_def_last=('SK_DPD_DEF', 'last'),
    pos_dpd_def_max=('SK_DPD_DEF', 'max'),
    pos_dpd_def_mean=('SK_DPD_DEF', 'mean'),
    pos_dpd_def_pct_overdue=('SK_DPD_DEF', lambda x: (x > 0).mean()),
).reset_index()

# Обработка категориальной переменной NAME_CONTRACT_STATUS
# Статус в последнем месяце
last_status = pc.groupby('SK_ID_PREV')['NAME_CONTRACT_STATUS'].last().reset_index()
last_status.columns = ['SK_ID_PREV', 'pos_last_status']

# Доля каждого статуса (one-hot encoding)
status_dummies = pd.get_dummies(pc['NAME_CONTRACT_STATUS'], prefix='pos_status')
pos_with_status = pd.concat([pc[['SK_ID_PREV']], status_dummies], axis=1)
status_agg = pos_with_status.groupby('SK_ID_PREV').mean().reset_index()

# Объединяем все агрегаты
pos_agg = pos_agg.merge(last_status, on='SK_ID_PREV', how='left')
pos_agg = pos_agg.merge(status_agg, on='SK_ID_PREV', how='left')

In [17]:
pos_agg.shape

(936325, 32)

In [18]:
# Сортировка для корректного взятия последних значений
ccb = ccb.sort_values(['SK_ID_PREV', 'MONTHS_BALANCE'])

# Агрегация основных числовых признаков
cc_agg = ccb.groupby('SK_ID_PREV').agg(
    # Количество месяцев
    cc_months_count=('MONTHS_BALANCE', 'count'),
    cc_first_month=('MONTHS_BALANCE', 'min'),
    cc_last_month=('MONTHS_BALANCE', 'max'),

    # Лимит
    cc_limit_last=('AMT_CREDIT_LIMIT_ACTUAL', 'last'),
    cc_limit_mean=('AMT_CREDIT_LIMIT_ACTUAL', 'mean'),
    cc_limit_max=('AMT_CREDIT_LIMIT_ACTUAL', 'max'),
    cc_limit_min=('AMT_CREDIT_LIMIT_ACTUAL', 'min'),

    # Баланс
    ccbance_last=('AMT_BALANCE', 'last'),
    ccbance_mean=('AMT_BALANCE', 'mean'),
    ccbance_max=('AMT_BALANCE', 'max'),
    ccbance_min=('AMT_BALANCE', 'min'),

    # Снятия ATM
    cc_drawings_atm_last=('AMT_DRAWINGS_ATM_CURRENT', 'last'),
    cc_drawings_atm_sum=('AMT_DRAWINGS_ATM_CURRENT', 'sum'),
    cc_drawings_atm_mean=('AMT_DRAWINGS_ATM_CURRENT', 'mean'),
    cc_cnt_drawings_atm_last=('CNT_DRAWINGS_ATM_CURRENT', 'last'),
    cc_cnt_drawings_atm_sum=('CNT_DRAWINGS_ATM_CURRENT', 'sum'),

    # Снятия POS
    cc_drawings_pos_last=('AMT_DRAWINGS_POS_CURRENT', 'last'),
    cc_drawings_pos_sum=('AMT_DRAWINGS_POS_CURRENT', 'sum'),
    cc_drawings_pos_mean=('AMT_DRAWINGS_POS_CURRENT', 'mean'),
    cc_cnt_drawings_pos_last=('CNT_DRAWINGS_POS_CURRENT', 'last'),
    cc_cnt_drawings_pos_sum=('CNT_DRAWINGS_POS_CURRENT', 'sum'),

    # Другие снятия
    cc_drawings_other_last=('AMT_DRAWINGS_OTHER_CURRENT', 'last'),
    cc_drawings_other_sum=('AMT_DRAWINGS_OTHER_CURRENT', 'sum'),
    cc_drawings_other_mean=('AMT_DRAWINGS_OTHER_CURRENT', 'mean'),
    cc_cnt_drawings_other_last=('CNT_DRAWINGS_OTHER_CURRENT', 'last'),
    cc_cnt_drawings_other_sum=('CNT_DRAWINGS_OTHER_CURRENT', 'sum'),

    # Общие снятия
    cc_drawings_total_last=('AMT_DRAWINGS_CURRENT', 'last'),
    cc_drawings_total_sum=('AMT_DRAWINGS_CURRENT', 'sum'),
    cc_drawings_total_mean=('AMT_DRAWINGS_CURRENT', 'mean'),
    cc_cnt_drawings_total_last=('CNT_DRAWINGS_CURRENT', 'last'),
    cc_cnt_drawings_total_sum=('CNT_DRAWINGS_CURRENT', 'sum'),

    # Платежи
    cc_payment_current_last=('AMT_PAYMENT_CURRENT', 'last'),
    cc_payment_current_sum=('AMT_PAYMENT_CURRENT', 'sum'),
    cc_payment_current_mean=('AMT_PAYMENT_CURRENT', 'mean'),
    cc_payment_total_last=('AMT_PAYMENT_TOTAL_CURRENT', 'last'),
    cc_payment_total_sum=('AMT_PAYMENT_TOTAL_CURRENT', 'sum'),
    cc_payment_total_mean=('AMT_PAYMENT_TOTAL_CURRENT', 'mean'),

    # Минимальный платёж
    cc_min_installment_last=('AMT_INST_MIN_REGULARITY', 'last'),
    cc_min_installment_mean=('AMT_INST_MIN_REGULARITY', 'mean'),

    # Дебиторская задолженность, счета к получению
    cc_receivable_principal_last=('AMT_RECEIVABLE_PRINCIPAL', 'last'),
    cc_receivable_principal_mean=('AMT_RECEIVABLE_PRINCIPAL', 'mean'),
    cc_receivable_last=('AMT_RECIVABLE', 'last'),
    cc_receivable_mean=('AMT_RECIVABLE', 'mean'),
    cc_total_receivable_last=('AMT_TOTAL_RECEIVABLE', 'last'),
    cc_total_receivable_mean=('AMT_TOTAL_RECEIVABLE', 'mean'),

    # Количество выплаченных платежей
    cc_instalment_mature_cum_last=('CNT_INSTALMENT_MATURE_CUM', 'last'),
    cc_instalment_mature_cum_max=('CNT_INSTALMENT_MATURE_CUM', 'max'),
    
).reset_index()

# Обработка категориального поля NAME_CONTRACT_STATUS
# Последний статус
last_status = ccb.groupby('SK_ID_PREV')['NAME_CONTRACT_STATUS'].last().reset_index()
last_status.columns = ['SK_ID_PREV', 'cc_last_status']

# Доля каждого статуса за весь период (one-hot)
status_dummies = pd.get_dummies(ccb['NAME_CONTRACT_STATUS'], prefix='cc_status')
cc_with_status = pd.concat([ccb[['SK_ID_PREV']], status_dummies], axis=1)
status_agg = cc_with_status.groupby('SK_ID_PREV').mean().reset_index()

# Объединяем все части
cc_agg = cc_agg.merge(last_status, on='SK_ID_PREV', how='left')
cc_agg = cc_agg.merge(status_agg, on='SK_ID_PREV', how='left')

cc_agg.shape

(104307, 56)

In [19]:
# Объединим агрегированные наборы в один набор по предыдущим кредитам
p1 = prev_app.merge(ip_agg, on='SK_ID_PREV', how='left')
p2 = p1.merge(cc_agg, on='SK_ID_PREV', how='left')
full_prev = p2.merge(pos_agg, on='SK_ID_PREV', how='left')

full_prev.shape

(1670214, 129)

Остаётся сгруппировать всю предыдущую информацию о кредитах по ключу SK_ID_CURR, а затем присоединить bureau и previous к application

In [20]:
# Определим словарь агрегаций для числовых колонок
numeric_agg = {
    # Признаки из previous_application
    'AMT_CREDIT': ['sum', 'mean', 'max'],
    'AMT_ANNUITY': ['sum', 'mean', 'max'],
    'AMT_GOODS_PRICE': ['sum', 'mean', 'max'],
    'AMT_DOWN_PAYMENT': ['sum', 'mean', 'max'],
    'CNT_PAYMENT': ['mean', 'max'],
    'DAYS_DECISION': ['mean', 'min'],
    'DAYS_FIRST_DRAWING': ['mean', 'min'],
    'DAYS_LAST_DUE': ['mean', 'min'],
    'RATE_DOWN_PAYMENT': ['mean'],
    'RATE_INTEREST_PRIMARY': ['mean'],
    'FLAG_LAST_APPL_PER_CONTRACT': ['sum', 'max'],
    'NFLAG_LAST_APPL_IN_DAY': ['sum', 'max'],
    
    # Признаки из installments_payments
    'sum_installment': ['sum'],
    'mean_installment': ['mean'],
    'sum_payment': ['sum'],
    'mean_payment': ['mean'],
    'days_max': ['max'],
    'days_entry_max': ['max'],
    
    # Признаки из credit_card_balance
    'cc_months_count': ['sum'],
    'cc_limit_last': ['mean'],
    'cc_limit_mean': ['mean'],
    'cc_limit_max': ['max'],
    'cc_limit_min': ['min'],
    'ccbance_last': ['mean'],
    'ccbance_mean': ['mean'],
    'ccbance_max': ['max'],
    'ccbance_min': ['min'],
    'cc_drawings_atm_sum': ['sum'],
    'cc_drawings_pos_sum': ['sum'],
    'cc_drawings_other_sum': ['sum'],
    'cc_drawings_total_sum': ['sum'],
    'cc_payment_current_sum': ['sum'],
    'cc_payment_total_sum': ['sum'],
    'cc_min_installment_last': ['mean'],
    'cc_receivable_principal_last': ['mean'],
    'cc_receivable_last': ['mean'],
    'cc_total_receivable_last': ['mean'],
    'cc_instalment_mature_cum_last': ['mean'],
    'cc_instalment_mature_cum_max': ['max'],
    
    # Признаки из pos_cash_balance
    'pos_months_count': ['sum'],
    'pos_instalment_last': ['mean'],
    'pos_future_last': ['mean'],
    'pos_future_diff': ['sum'], 
    'pos_dpd_max': ['max'],
    'pos_dpd_mean': ['mean'],
    'pos_dpd_pct_overdue': ['mean'],
    'pos_dpd_def_max': ['max'],
    'pos_dpd_def_mean': ['mean'],
    'pos_dpd_def_pct_overdue': ['mean'],
}

# Для категориальных колонок со статусами (доли)
categorical_status_cols = [col for col in full_prev.columns if col.startswith('cc_status_') or col.startswith('pos_status_')]

# Для последних статусов (категориальные строки)
last_status_cols = ['cc_last_status', 'pos_last_status']

# Выполняем группировку
grouped = full_prev.groupby('SK_ID_CURR')

# Числовая агрегация
numeric_result = grouped.agg(numeric_agg)

# Упрощаем мультииндекс
numeric_result.columns = ['_'.join(col).strip() for col in numeric_result.columns.values]

# Агрегация для долей статусов (среднее значение = доля)
status_dummies_agg = grouped[categorical_status_cols].mean().add_suffix('_avg')

# Для последних статусов: берём последнее значение по какому-то критерию.
# Так как у нас нет временной привязки между разными SK_ID_PREV, можно взять моду или последний по дате,
# если есть поле с датой. Упрощённо: возьмём моду (наиболее часто встречающийся статус).
# Для этого нужно сначала определить моду для каждой группы.
def mode_series(x):
    return x.mode().iloc[0] if not x.mode().empty else None

last_status_agg = grouped[last_status_cols].agg(mode_series).add_suffix('_mode')

# Объединяем все результаты
final_df = numeric_result.join(status_dummies_agg).join(last_status_agg).reset_index()

# При необходимости добавим количество предыдущих кредитов
final_df['n_previous_apps'] = grouped.size().values

In [21]:
final_df.shape

(338857, 83)

In [22]:
# Сохраним данные
final_df.to_csv('../../data/clean/previous.csv', index=False)

In [23]:
# Объединим данные в один набор
app = pd.read_csv('../../data/clean/application_clean.csv', index_col=0)
bureau = pd.read_csv('../../data/clean/bureau_agg.csv', index_col=0)

app_plus_bureau = app.merge(bureau, on='SK_ID_CURR', how='left')
full_data = app_plus_bureau.merge(final_df, on='SK_ID_CURR', how='left')

# Посмотрим размер
full_data.shape

(307511, 215)

In [24]:
# Сохраним полный набор
full_data.to_csv('../../data/clean/combined_data.csv', index=False)

In [25]:
data = pd.read_csv('../../data/clean/combined_data.csv', index_col=0)

In [26]:
columns = data.columns.tolist()
columns

['TARGET',
 'NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'OWN_CAR_AGE',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'ORGANIZATION_TYPE',
 'APARTMENTS_AVG',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_AVG',
 'YEARS_BUILD_AVG',
 'COMMONAREA_AVG',
 'ELEVATORS_AVG',
 'ENTRANCES_AVG',
 'FLOORSMAX_AVG',
 'FLOORSMIN_AVG',


In [27]:
application_cols = [
    'TARGET',
    'NAME_CONTRACT_TYPE',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'OCCUPATION_TYPE',
    'AGE',
    'YEARS_EMPLOYED',
    'has_children',
    'FLAG_MOBIL',
]
len(application_cols)

13

In [28]:
application_dataset = data[application_cols]
application_dataset.head()

,TARGET,NAME_CONTRACT_TYPE,FLAG_OWN_CAR,FLAG_OWN_REALTY,AMT_INCOME_TOTAL,AMT_CREDIT,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,OCCUPATION_TYPE,AGE,YEARS_EMPLOYED,has_children,FLAG_MOBIL
SK_ID_CURR,,,,,,,,,,,,,
100002,1,Cash loans,N,Y,202500.0,406597.5,Secondary / secondary special,Single / not married,Laborers,25,1,0,1
100003,0,Cash loans,N,N,270000.0,1293502.5,Higher education,Married,Core staff,45,3,0,1
100004,0,Revolving loans,Y,Y,67500.0,135000.0,Secondary / secondary special,Single / not married,Laborers,52,0,0,1
100006,0,Cash loans,N,Y,135000.0,312682.5,Secondary / secondary special,Civil marriage,Laborers,52,8,0,1
100007,0,Cash loans,N,Y,121500.0,513000.0,Secondary / secondary special,Single / not married,Core staff,54,8,0,1


In [29]:
application_dataset.to_csv('../../data/clean/first_stage_data.csv', index=False)